# STEP 1: SFS

# Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive


# Move to the Dataset Directory in My Drive

In [ ]:
import os
os.chdir("/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025/Sample Data")
!pwd

/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025/Sample Data


# SFS Code:

In [ ]:
# importing necessary packages
import matplotlib.pyplot as plt # for making plots / graphs
import pandas            as pd  # for reading the .csv file and related operations
import numpy             as np  # for working with arrays (multi-dimensional)
import os
from sklearn.model_selection   import StratifiedKFold
from sklearn.svm               import SVC
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.metrics           import accuracy_score, f1_score, matthews_corrcoef
from sklearn.preprocessing     import StandardScaler
from utils                  import *

# CARGAR LA BASE DE DATOS
file_path = f"ALL FEATURES OLDER - Pes Planus and Control JUNE 25 Train.xlsx"
EXCEL_file = pd.ExcelFile(file_path)
Step_1_ML = {}

for sheet_name in EXCEL_file.sheet_names:
    # Load the sheet into a DataFrame
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    # VALIDAR QUE EL CARGADO DE DATOS NO HAYA MODIFICADO EL DATASET
    print('Columns before:', df.shape)
    print('Columns After:',  df.shape)

    # DEFINIR EL CAMBIO DE VARIABLE EN LA CLASIFICACIÓN OBJETIVA -y
    df["Group"] = df["Group"].replace({'Control': 0, 'Flatfoot': 1})
    # saving the target variables into `y` variable.
    y = df.loc[:, "Group"].values
    df["Group"]
    # Define the different segments from dataset to be used.
    segments = {
        'All_Data': df.loc[:, 'Pelv_Angle_Y_MAX_SW': 'Hip_Angle_Z_OHS'],
    }
    # define scaler
    sc = StandardScaler()

    # define crossvalidation
    kfold_cv = StratifiedKFold(
                    n_splits     = 10,
                    shuffle      = True,
                    random_state = 0
                    )

    # Defining all combinations of C and gamma...

    C_set = [0.1, 1, 25, 50, 75, 100, 125, 150]
    r_set = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]

    algorith_you_are_using = 'SVM (rbf)'
    y_true_list = []
    y_pred_list = []
    df_results = pd.DataFrame(columns=['CV_Accuracy', 'C', 'Gamma', '#Features', 'Ordered Indices'])
    N_FEATS = 40
    counter = 0
    for C_val in C_set:
        for r_val in r_set:

            X = segments["All_Data"].values
            # fit a scaler to entire data
            X = sc.fit_transform(X)

            svm = SVC(kernel  = 'rbf',
                    C       = C_val,
                    gamma   = r_val,
                    verbose = False)

            sfs = SFS(estimator  = svm,
                    k_features = N_FEATS,
                    forward    = True,
                    floating   = False,
                    verbose    = 1,
                    scoring    = ('accuracy'),
                    cv         = kfold_cv,
                    n_jobs     = -1)

            # Apply SFS
            # sfs.k_features = (1, X.shape[1])
            sfs.fit(X, y)
            sfs_results = sfs.get_metric_dict()

            selected_features_order = []
            setA = []
            for i in range(1, N_FEATS + 1):
                setB = setA
                setA = set(sfs_results[i]['feature_names'])
                newfeat = setA.difference(setB)
                selected_features_order.append(int(newfeat.pop()))
            print("selected_features_order", selected_features_order)

            # Apply Grid Search on the Most Significant Features
            X = sfs.transform(X)

            # Apply kfoldCV to get classification scores
            y_true_unflat_list, y_pred_unflat_list = [], []
            for train_idx, test_idx in kfold_cv.split(X, y):
                x_train, y_train = X[train_idx], y[train_idx]
                x_test, y_test   = X[test_idx], y[test_idx]

                svm.fit(x_train, y_train)

                y_pred = svm.predict(x_test)

                y_true_unflat_list.append(y_test[:])
                y_pred_unflat_list.append(y_pred[:])

                y_true_list = [item for sublist in y_true_unflat_list for item in sublist]
                y_pred_list = [item for sublist in y_pred_unflat_list for item in sublist]

            # if accuracy_score(y_true_list, y_pred_list) >= 0.75:
            print("{}, {}, {}, {}".format(accuracy_score(y_true_list, y_pred_list), C_val, r_val, len(sfs.k_feature_idx_)))
            counter += 1
            print(counter)
            print("\n################################################\n")
            df_results = pd.concat([df_results, pd.DataFrame(
                [[accuracy_score(y_true_list, y_pred_list), C_val, r_val, len(sfs.k_feature_idx_),
                selected_features_order]],
                columns=['CV_Accuracy', 'C', 'Gamma', '#Features', 'Ordered Indices'])], ignore_index=True)

            Step_1_ML[sheet_name] = df_results

output_path = "ML_Step_1.xlsx"
# Save all sheets to a new Excel file
with pd.ExcelWriter(output_path) as writer:
    for sheet_name, df_filtered in Step_1_ML.items():
        df_filtered.to_excel(writer, sheet_name=sheet_name, index=False)